<!-- cabecera-entorno -->
## Antes de empezar

**Cinco variaciones modernas del diagrama de caja y bigotes** — EVA 2019–2025 (UPRA · datos.gov.co)

Ejecute el cuaderno **de arriba a abajo, en orden**. La celda siguiente verifica el entorno: si algo
falla ahí, no siga. Cada figura se dibuja **dentro del cuaderno** al ejecutar su celda (`fig.show()`),
y ademas se exporta a HTML interactivo y PNG.

**Tres errores frecuentes**

1. **Kernel equivocado.** Si usa el Python del sistema en vez del entorno `.venv`, falla un `import`.
   Elija el kernel `.venv` en la esquina superior derecha.
2. **Falta el crudo.** Si no esta descargado el CSV en `data/raw/`, la primera celda de trabajo lo
   intenta bajar de la API; sin internet, délo por fallido y revise la ruta.
3. **Ejecutar en desorden.** Las figuras usan las tablas del corte; si salta celdas, esas variables
   no existen cuando se dibuja.

In [ ]:
# Verificacion del entorno. Si algo falla aqui, no siga: la salida dice que falta.
import sys
from pathlib import Path

print("Interprete :", sys.executable)
print("Python     :", sys.version.split()[0])
faltan = []
for mod in ["pandas", "numpy", "plotly", "scipy", "kaleido"]:
    try:
        _m = __import__(mod)
        print(f"  {mod:8s}", getattr(_m, "__version__", "?"))
    except Exception as error:
        faltan.append(mod)
        print(f"  {mod:8s} FALTA:", error)


def _raiz_del_proyecto(inicio=None):
    # Sube desde el directorio actual hasta hallar el repo (tiene 'data/raw' y 'proyecto_boxplots').
    inicio = Path(inicio or Path.cwd()).resolve()
    for base in (inicio, *inicio.parents):
        if (base / "data" / "raw").is_dir() and (base / "proyecto_boxplots").is_dir():
            return base
    return Path.cwd()


RUTA_CSV_CHEQUEO = (_raiz_del_proyecto()
                    / "data/raw/Evaluaciones_Agropecuarias_Municipales_EVA_2019_2025_agricola.csv")
print("Raiz proyecto:", _raiz_del_proyecto())
print("CSV crudo  :", RUTA_CSV_CHEQUEO.exists(), "->", RUTA_CSV_CHEQUEO.name)
print("\nEntorno OK" if not faltan and RUTA_CSV_CHEQUEO.exists()
      else "\nREVISAR: hay librerias o el CSV pendientes (ver README: Cómo se ejecuta)")


# Cinco variaciones modernas del diagrama de caja y bigotes

## Rendimiento agrícola municipal — EVA 2019–2025 (UPRA · datos.gov.co)

**Pregunta conductora:** ¿Qué tan desigual es el rendimiento de un mismo cultivo entre municipios,
y qué revela cada gráfico que el boxplot esconde?

**Dataset:** *Evaluaciones Agropecuarias Municipales – EVA. 2019 - 2025. Base Agrícola* (UPRA).
Enlace directo: <https://www.datos.gov.co/d/uejq-wxrr>

**Unidad de análisis:** un registro = municipio × desagregación de cultivo × periodo. Cada punto de
las figuras es un registro municipal; **no** es un promedio ponderado. El rendimiento se mide en
t/ha y **no es comparable entre cultivos** (el café rinde ~1 t/ha; el tomate ~25), por lo que se
compara dentro de un cultivo o se normaliza.

## 0 · Configuración compartida

Importaciones, rutas y **paleta única** (un acento sobre gris, legible en escala de grises). Todas
las figuras usan esta paleta y ordenan las categorías por mediana, no alfabéticamente.

In [ ]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio


def hallar_raiz_proyecto(inicio=None):
    # Raiz del repo = carpeta que contiene 'proyecto_boxplots/' y 'data/raw/'.
    # Portable: no depende de la ruta absoluta de una maquina concreta.
    if os.environ.get("PROYECTO_RAIZ"):
        return Path(os.environ["PROYECTO_RAIZ"]).expanduser().resolve()
    inicio = Path(inicio or Path.cwd()).resolve()
    for base in (inicio, *inicio.parents):
        if (base / "proyecto_boxplots").is_dir() and (base / "data" / "raw").is_dir():
            return base
    raise FileNotFoundError(
        "No se encontro la raiz del proyecto (carpeta con 'proyecto_boxplots/' y 'data/raw/'). "
        "Ejecute el cuaderno desde la raiz del repositorio o defina la variable PROYECTO_RAIZ."
    )


RAIZ = hallar_raiz_proyecto()
PROY = RAIZ / "proyecto_boxplots"
DIR_HTML = PROY / "figuras_interactivas"
DIR_PNG = PROY / "figuras"
DIR_HTML.mkdir(parents=True, exist_ok=True)
DIR_PNG.mkdir(parents=True, exist_ok=True)
RUTA_CSV = RAIZ / "data" / "raw" / "Evaluaciones_Agropecuarias_Municipales_EVA_2019_2025_agricola.csv"
RUTA_BITACORA = RAIZ / "data" / "raw" / "bitacora_descarga.json"

# Paleta unica: un acento sobre gris (ColorBrewer Grays, segura en escala de grises)
ACENTO = "#0b5394"
GRISES = ["#252525", "#525252", "#737373", "#969696", "#bdbdbd", "#d9d9d9"]
TEXTO = "#202124"
FUENTE_FIG = ("Fuente: EVA 2019-2025, UPRA (datos.gov.co/d/uejq-wxrr). "
              "Cada punto = municipio x desagregacion x periodo. Elaboracion propia.")
pio.templates.default = "plotly_white"

# Contenedor de interpretaciones (se llena por figura y se exporta para la presentacion)
INTERPRETACIONES = {}

def exportar_figura(figura, nombre, ancho, alto):
    """Exporta siempre el HTML autónomo y el PNG cuando Kaleido encuentre Chrome."""
    ruta_html = DIR_HTML / f"{nombre}.html"
    ruta_png = DIR_PNG / f"{nombre}.png"
    figura.write_html(ruta_html, include_plotlyjs=True, full_html=True)
    try:
        figura.write_image(ruta_png, width=ancho, height=alto, scale=2)
        estado_png = f"generado ({ruta_png.stat().st_size / 1024:.1f} KB)"
    except Exception as error:
        estado_png = "NO generado"
        print("ADVERTENCIA PNG: Kaleido necesita Chrome para exportar imágenes. "
              "Ejecute una vez `.venv/bin/plotly_get_chrome -y` y vuelva a correr el cuaderno.")
        print(f"  Detalle: {type(error).__name__}: {error}")
    print(f"HTML autónomo: {ruta_html.name} | {ruta_html.stat().st_size / 1024:.1f} KB")
    print(f"PNG: {ruta_png.name} | {estado_png}")
    return ruta_html, ruta_png

print("Raiz proyecto:", RAIZ)
print("Configuracion lista. Acento:", ACENTO, "| Fuente citada en figuras:", bool(FUENTE_FIG))


## 1 · Ingesta y perfil

Descarga reproducible por la API Socrata (idempotente: solo si falta el crudo), carga y limpieza
declarada. Se imprimen URL, fecha de descarga, filas, columnas, años y periodos presentes, y se
verifica la unicidad de la unidad de análisis.

In [ ]:
URL_HUMANA = "https://www.datos.gov.co/d/uejq-wxrr"
URL_API = "https://www.datos.gov.co/resource/uejq-wxrr.json"
COUNT_API = 166732


def descargar_si_falta():
    if RUTA_CSV.exists():
        print("Crudo ya presente, no se re-descarga:", RUTA_CSV.name)
        return
    import requests
    sesion = requests.Session()
    filas, offset, LIM = [], 0, 50000
    while True:
        r = sesion.get(URL_API, params={"$limit": LIM, "$offset": offset, "$order": ":id"}, timeout=180)
        r.raise_for_status()
        lote = r.json()
        if not lote:
            break
        filas.extend(lote)
        offset += LIM
        if len(lote) < LIM:
            break
        time.sleep(0.3)
    pd.DataFrame(filas).to_csv(RUTA_CSV, index=False, encoding="utf-8")
    print("Descargadas", len(filas), "filas en", URL_API)


descargar_si_falta()

REN = {
    "c_digo_dane_departamento": "codigo_dane_departamento", "departamento": "departamento",
    "c_digo_dane_municipio": "codigo_dane_municipio", "municipio": "municipio",
    "grupo_cultivo": "grupo_cultivo", "subgrupo": "subgrupo", "cultivo": "cultivo",
    "desagregaci_n_cultivo": "desagregacion_cultivo", "a_o": "anio", "periodo": "periodo",
    "rea_sembrada": "area_sembrada", "rea_cosechada": "area_cosechada", "producci_n": "produccion",
    "rendimiento": "rendimiento", "ciclo_del_cultivo": "ciclo_del_cultivo",
    "estado_f_sico_del_cultivo": "estado_fisico_del_cultivo", "c_digo_del_cultivo": "codigo_cultivo",
    "nombre_cient_fico_del_cultivo": "nombre_cientifico_cultivo",
}
df = pd.read_csv(RUTA_CSV, dtype=str, keep_default_na=False, encoding="utf-8").rename(columns=REN)
for c in ["area_sembrada", "area_cosechada", "produccion", "rendimiento"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["anio"] = pd.to_numeric(df["anio"], errors="coerce").astype("Int64")
df["tipo_periodo"] = df["periodo"].str.extract(r"(\d{4})([AB]?)$", expand=False)[1].replace("", "anual")

bit = json.loads(RUTA_BITACORA.read_text(encoding="utf-8"))
print("URL (datos)         :", URL_HUMANA)
print("URL (API)           :", URL_API)
print("Fecha de descarga   :", bit["fecha_descarga_utc"])
print("Filas crudas        :", len(df), "| count(*):", COUNT_API, "| coincide:", len(df) == COUNT_API)
print("Columnas            :", df.shape[1])
print("Años presentes      :", sorted(df["anio"].unique().tolist()))
print("Periodos presentes  :", df["periodo"].nunique(), "->", sorted(df["periodo"].unique().tolist()))
print("Tipos de periodo    :", df["tipo_periodo"].value_counts().to_dict())

LLAVE = ["codigo_dane_municipio", "desagregacion_cultivo", "periodo"]
print("Filas duplicadas en la unidad (municipio x desagregacion x periodo):",
      int(df.duplicated(subset=LLAVE).sum()))

## 2 · Figura 1 — Half-Eye Plot · papa por departamento

**Concepto.** El boxplot reduce la distribución a cinco números y dibuja la caja entre Q1 y Q3. El
*Half-Eye Plot* (una mitad del *eye plot*) reemplaza la caja por la **densidad real** del dato: una
KDE reflejada en el eje, la mediana marcada, y **intervalos del 66 % y 95 %** como bandas. Corrige
la falla central del boxplot: la caja no dice *cómo* se reparte la masa dentro de los cuartiles ni
dónde están los huecos y las concentraciones.

**Uso recomendado.** Es útil cuando se quieren comparar **pocas o varias categorías** con una
variable continua, para mostrar la **forma de la distribución**, el **centro** (mediana) y su
dispersión. Aquí se usa para contrastar cinco departamentos: permite ver dónde está la masa, si hay
colas largas y cómo varían las formas entre grupos. No es necesario que las categorías sean muchas;
lo importante es que el corte tenga datos suficientes por grupo.

**Nota sobre los intervalos.** Los intervalos que se dibujan son **cuantiles empíricos** de la
distribución observada: 66 % → [p17, p83], 95 % → [p2.5, p97.5] (`np.quantile`). Son **no**
intervalos de credibilidad bayesianos ni bandas de error muestral; reflejan la dispersión del dato
observado, no una estimación de un parámetro poblacional.

**Paso (a): tabla del corte.** Se filtra papa, se descartan los registros sin cosecha
(rendimiento = 0) y se retienen cinco departamentos. Cada filtro reporta cuántas filas quita.

In [ ]:
print("=== Figura 1 - Half-Eye: papa por departamento ===")
print("Pregunta: que tan disperso es el rendimiento de la papa entre municipios de cada depto,")
print("y donde esta la masa frente a los extremos que la caja del boxplot no muestra?")

DEPTOS_PAPA = ["Boyacá", "Cundinamarca", "Nariño", "Antioquia", "Santander"]

# Filtro 1: cultivo == Papa
m1 = df["cultivo"] == "Papa"
print(f"Filtro 1 cultivo=='Papa'      : conserva {int(m1.sum()):6d} de {len(df):6d} (quita {int((~m1).sum()):6d})")
paso = df[m1]
# Filtro 2: rendimiento > 0 (rendimiento 0 = sin cosecha, no es una medicion)
m2 = paso["rendimiento"] > 0
print(f"Filtro 2 rendimiento>0        : conserva {int(m2.sum()):6d} de {len(paso):6d} (quita {int((~m2).sum()):6d})")
paso = paso[m2]
# Filtro 3: departamentos objetivo
m3 = paso["departamento"].isin(DEPTOS_PAPA)
print(f"Filtro 3 deptos objetivo      : conserva {int(m3.sum()):6d} de {len(paso):6d} (quita {int((~m3).sum()):6d})")
papa = paso[m3].copy()

tabla_papa = (papa.groupby("departamento")
              .agg(N=("rendimiento", "size"),
                   municipios=("codigo_dane_municipio", "nunique"),
                   mediana=("rendimiento", "median"),
                   p25=("rendimiento", lambda s: s.quantile(0.25)),
                   p75=("rendimiento", lambda s: s.quantile(0.75)))
              .reindex(DEPTOS_PAPA))
tabla_papa["N_por_100muestras"] = (tabla_papa["N"] / tabla_papa["N"].sum() * 100).round(1)
print("\nTabla del corte (Half-Eye), orden por mediana:")
print(tabla_papa.sort_values("mediana", ascending=False).round(2).to_string())
print("\nComprobacion: N minimo por grupo =", int(tabla_papa["N"].min()),
      "| N total =", int(tabla_papa["N"].sum()),
      "| grupos con N>0 =", int((tabla_papa["N"] > 0).sum()), "de", len(DEPTOS_PAPA))

**Pregunta (Figura 1):** ¿qué tan disperso es el rendimiento de la papa entre los municipios de cada
departamento, y dónde está la masa frente a los extremos que la caja del boxplot no muestra?

**Título conclusivo (escrito antes de dibujar, se valida en el paso c):**
*"El rendimiento de la papa separa a Cundinamarca y Boyacá de Antioquia y Nariño por la mediana,
pero dentro de cada departamento la dispersión supera los 10 t/ha."*

**Paso (b): figura interactiva + contraste.** Plotly no trae el Half-Eye como función: se construye
a mano con una KDE por grupo (evaluada solo dentro del rango observado), una mediana y los
intervalos del 66 % y 95 % (cuantiles 17/83 y 2.5/97.5). Se dibuja **junto** el `go.Box` del mismo
corte. Se exporta a HTML autónomo y PNG.

In [ ]:
from scipy.stats import gaussian_kde
from plotly.subplots import make_subplots

# --- Titulo conclusivo escrito ANTES de dibujar (se valida en el paso c) ---
TITULO_F1 = ("Papa: medianas de 14 a 20 t/ha; amplitud del intervalo central 95% de 22 a 37 t/ha")

orden_papa = tabla_papa.sort_values("mediana").index.tolist()  # ascendente por mediana
H = 0.9
etiquetas_papa = [f"{d}  (n={int(tabla_papa.loc[d, 'N'])})" for d in orden_papa]

def mitad_eye(v, y0, color_relleno="rgba(11,83,148,0.35)"):
    # Densidad (KDE) dentro del rango observado, escalada a altura H.
    grid = np.linspace(v.min(), v.max(), 200)
    dens = gaussian_kde(v)(grid)
    return grid, dens / dens.max() * H

# Los paneles usan ejes y independientes: el Half-Eye es numerico y el contraste es categorico.
fig1 = make_subplots(rows=1, cols=2, shared_yaxes=False, column_widths=[0.62, 0.38],
                     horizontal_spacing=0.04,
                     subplot_titles=("Half-Eye · densidad KDE + mediana + intervalos 66% y 95%",
                                     "Boxplot tradicional (mismo corte)"))

for i, d in enumerate(orden_papa):
    sub = papa[papa["departamento"] == d]
    v = sub["rendimiento"].to_numpy()
    grid, dens = mitad_eye(v, i)
    q025, q17, q50, q83, q975 = np.quantile(v, [0.025, 0.17, 0.5, 0.83, 0.975])
    # (1) densidad rellena (media eye)
    fig1.add_trace(go.Scatter(
        x=np.concatenate([grid, grid[::-1]]),
        y=np.concatenate([i + dens, np.full_like(grid, i)]),
        fill="toself", fillcolor="rgba(11,83,148,0.30)", line=dict(color=ACENTO, width=1),
        hoverinfo="text", showlegend=False,
        hovertext=f"{d}<br>n = {len(v)}<br>mediana = {q50:.1f} t/ha<br>IQR central 66% = {q17:.1f}-{q83:.1f}"),
        row=1, col=1)
    # (2) intervalos 95% y 66%
    fig1.add_trace(go.Scatter(x=[q025, q975], y=[i, i], mode="lines",
        line=dict(color=GRISES[2], width=3), hoverinfo="text", showlegend=False,
        hovertext=f"{d} · intervalo 95%: {q025:.1f}-{q975:.1f} t/ha"), row=1, col=1)
    fig1.add_trace(go.Scatter(x=[q17, q83], y=[i, i], mode="lines",
        line=dict(color=GRISES[0], width=6), hoverinfo="text", showlegend=False,
        hovertext=f"{d} · intervalo 66%: {q17:.1f}-{q83:.1f} t/ha"), row=1, col=1)
    # (3) mediana
    fig1.add_trace(go.Scatter(x=[q50], y=[i], mode="markers",
        marker=dict(color=ACENTO, size=11, symbol="diamond", line=dict(color="white", width=1)),
        hoverinfo="text", showlegend=False,
        hovertext=f"{d} · mediana: {q50:.1f} t/ha"), row=1, col=1)
    # (4) puntos crudos con jitter fijo -> hover por registro (municipio, depto, cultivo, periodo, rendimiento, n)
    rng = np.random.default_rng(20261014)
    jit = rng.uniform(0, 0.16, len(v))
    hov = [f"Municipio: {mu}<br>Departamento: {d}<br>Cultivo: {cu}<br>Periodo: {pe}"
           f"<br>Rendimiento: {r:.1f} t/ha<br>n del departamento: {len(v)}"
           for mu, cu, pe, r in zip(sub["municipio"], sub["cultivo"], sub["periodo"], v)]
    fig1.add_trace(go.Scatter(x=v, y=i + jit, mode="markers",
        marker=dict(color=ACENTO, size=3, opacity=0.22), hoverinfo="text",
        hovertext=hov, showlegend=False), row=1, col=1)

# Boxplot tradicional del MISMO corte
for d in orden_papa:
    v = papa.loc[papa["departamento"] == d, "rendimiento"]
    fig1.add_trace(go.Box(x=v, name=d, orientation="h",
        marker_color=GRISES[1], fillcolor="rgba(150,150,150,0.35)",
        line=dict(color=GRISES[1]), boxpoints="outliers",
        marker=dict(color=GRISES[1], size=3), showlegend=False,
        hovertemplate=f"{d}<br>%{{x:.1f}} t/ha<extra></extra>"), row=1, col=2)

# n visible en las etiquetas, sin superponer anotaciones sobre los datos.
fig1.update_yaxes(tickmode="array", tickvals=list(range(len(orden_papa))), ticktext=etiquetas_papa,
                  title_text="Departamento (orden por mediana)",
                  range=[-0.35, len(orden_papa) - 1 + H + 0.35], row=1, col=1)
fig1.update_yaxes(type="category", categoryorder="array", categoryarray=orden_papa,
                  tickmode="array", tickvals=orden_papa, ticktext=etiquetas_papa,
                  title_text="Departamento (orden por mediana)", row=1, col=2)
fig1.update_xaxes(title_text="Rendimiento (t/ha) - escala lineal", row=1, col=1)
fig1.update_xaxes(title_text="Rendimiento (t/ha) - escala lineal", row=1, col=2)
fig1.update_layout(title=dict(text=TITULO_F1, font=dict(size=15)), width=1180, height=560,
    margin=dict(l=60, r=30, t=90, b=80), plot_bgcolor="white", paper_bgcolor="white")
fuente_ann = dict(text=FUENTE_FIG, showarrow=False, xref="paper", yref="paper",
                  x=0, y=-0.14, xanchor="left", font=dict(size=10, color=TEXTO))
fig1.update_layout(annotations=list(fig1.layout.annotations) + [fuente_ann])

exportar_figura(fig1, "fig1_halfeye_papa", 1180, 560)
print("Traces en la figura:", len(fig1.data))

fig1.show()   # dibuja la figura dentro del cuaderno (como plt.show())

**Paso (c): interpretación y validación del título.** Se calculan los estadísticos por departamento y
el párrafo se arma con *f-strings* desde esas variables (ninguna cifra se escribe a mano). Después se
**valida el título conclusivo escrito antes de dibujar** contra la tabla: si no se sostiene, se
reporta la discrepancia en lugar de retocar el título.

In [ ]:
# === Figura 1 · Half-Eye — Paso (c): interpretacion desde variables ===
resumen = (papa.groupby("departamento")["rendimiento"]
           .agg(n="size", minimo="min",
                p025=lambda s: s.quantile(0.025), p17=lambda s: s.quantile(0.17),
                mediana="median", media="mean",
                p83=lambda s: s.quantile(0.83), p975=lambda s: s.quantile(0.975),
                maximo="max")
           .sort_values("mediana", ascending=False))
resumen["ancho_ic95"] = resumen["p975"] - resumen["p025"]
resumen["ancho_ic66"] = resumen["p83"] - resumen["p17"]
resumen["media_menos_mediana"] = resumen["media"] - resumen["mediana"]


def frac_outliers_regla_caja(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return float(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).mean())


outlier_por_depto = papa.groupby("departamento")["rendimiento"].apply(frac_outliers_regla_caja)

alto, bajo = resumen.index[0], resumen.index[-1]
m_alto, m_bajo = resumen.loc[alto, "mediana"], resumen.loc[bajo, "mediana"]
brecha = m_alto - m_bajo
mult = m_alto / m_bajo
disp_min, disp_max = resumen["ancho_ic95"].min(), resumen["ancho_ic95"].max()
depto_disp_max = resumen["ancho_ic95"].idxmax()
n_cola_alta = int((resumen["media_menos_mediana"] > 0).sum())
outl_pct = float((papa.groupby("departamento")["rendimiento"].apply(frac_outliers_regla_caja)
                  * papa.groupby("departamento").size()).sum() / len(papa) * 100)
outl_n = int(round(outl_pct / 100 * len(papa)))

texto_f1 = (
    f"En papa, las medianas departamentales van de {bajo} ({m_bajo:.1f} t/ha) a "
    f"{alto} ({m_alto:.1f} t/ha); la brecha entre extremos es de {brecha:.1f} t/ha "
    f"({mult:.2f}x). La forma de las densidades del Half-Eye muestra que la masa no es "
    f"simetrica: en {n_cola_alta} de {len(resumen)} departamentos la media supera la mediana, "
    f"una cola alta empujada por municipios de rendimiento excepcional. La dispersion interna es "
    f"grande en todos: el ancho del intervalo del 95% va de {disp_min:.1f} a {disp_max:.1f} t/ha, "
    f"el mayor en {depto_disp_max}. El boxplot tradicional marcaria como atipicos, en conjunto, "
    f"{outl_pct:.1f}% de los registros ({outl_n} de {len(papa)}); el Half-Eye deja ver que buena "
    f"parte de esa cola no son errores sino la forma real del rendimiento. El analisis es "
    f"descriptivo: no se atribuye ninguna causa a las diferencias."
)
INTERPRETACIONES["fig1_halfeye"] = texto_f1
print(texto_f1)

print("\n--- Validacion del titulo conclusivo (escrito antes de dibujar) ---")
v_mediana = bool(np.isclose(resumen["mediana"].min(), 14.0) and np.isclose(resumen["mediana"].max(), 20.0))
v_disp = bool(np.isclose(resumen["ancho_ic95"].min(), 22.0) and np.isclose(resumen["ancho_ic95"].max(), 37.0))
print("  'medianas de 14 a 20 t/ha'                         :", v_mediana,
      "->", list(resumen["mediana"].sort_values().round(1)))
print("  'amplitud del intervalo 95% de 22 a 37 t/ha'       :", v_disp,
      "->", list(resumen["ancho_ic95"].round(1)))
print("  Titulo sostenido por la tabla:", bool(v_mediana and v_disp))

import json
(DIR_HTML.parent / "interpretaciones.json").write_text(
    json.dumps(INTERPRETACIONES, ensure_ascii=False, indent=2), encoding="utf-8")
print("Interpretaciones guardadas en interpretaciones.json")

## 3 · Figura 2 — Ridge Plot (gráfico de crestas) · plátano por año

**Concepto.** El *Ridge Plot* apila las densidades de muchos grupos sobre un **mismo eje x**,
desplazándolas en y, de modo que se comparan las **formas** sin que unas tapen a otras. Corrige la
limitación del boxplot de tener que elegir entre mostrar muchos grupos (perdiendo la forma) o pocos
(perdiendo la vista de conjunto).

**Uso recomendado.** Comparar la distribución de una variable a lo largo de **muchas** categorías
(años, sitios, tratamientos) cuando importa la forma —dónde está la masa, si hay colas o
bimodalidad—, con N suficiente por grupo. **No** conviene con pocos grupos (mejor Half-Eye o violín)
ni cuando la densidad se extrapola fuera del rango observado.

**Paso (a): tabla del corte.** Se filtra plátano (convención anual), se descartan los registros sin
cosecha y se tabula N por año. El eje x será común a todas las crestas.

In [ ]:
print("=== Figura 2 - Ridge: platano por anio (2019-2025) ===")
print("Pregunta: como se mueve la forma de la distribucion del rendimiento del platano entre")
print("municipios, anio a anio, si el desplazamiento es de mediana o de dispersion?")

m1 = df["cultivo"] == "Plátano"
print(f"Filtro 1 cultivo=='Platano' : conserva {int(m1.sum()):6d} de {len(df):6d} (quita {int((~m1).sum()):6d})")
paso2 = df[m1]
m2 = paso2["rendimiento"] > 0
print(f"Filtro 2 rendimiento>0      : conserva {int(m2.sum()):6d} de {len(paso2):6d} (quita {int((~m2).sum()):6d})")
platano = paso2[m2].copy()

tabla_platano = (platano.groupby("anio")
                 .agg(N=("rendimiento", "size"),
                      municipios=("codigo_dane_municipio", "nunique"),
                      mediana=("rendimiento", "median"),
                      p25=("rendimiento", lambda s: s.quantile(0.25)),
                      p75=("rendimiento", lambda s: s.quantile(0.75)))
                 .sort_index())
print("\nTabla del corte (Ridge), por anio:")
print(tabla_platano.round(2).to_string())

xmin_pl, xmax_pl = float(platano["rendimiento"].min()), float(platano["rendimiento"].max())
p01, p99 = float(platano["rendimiento"].quantile(0.01)), float(platano["rendimiento"].quantile(0.99))
print(f"\nDominio x comun (todas las crestas): [{xmin_pl:.1f}, {xmax_pl:.1f}] t/ha")
print(f"  Percentiles 1-99: [{p01:.1f}, {p99:.1f}]")
print(f"Comprobacion: anios = {tabla_platano.index.tolist()} | N minimo/anio = {int(tabla_platano['N'].min())} "
      f"| grupos con N>0 = {int((tabla_platano['N'] > 0).sum())} de {len(tabla_platano)} "
      f"| convencion de periodo = {platano['tipo_periodo'].unique().tolist()}")

**Pregunta (Figura 2):** ¿cómo se mueve la forma de la distribución del rendimiento del plátano entre
municipios, año a año, y si el cambio es de mediana o de dispersión?

**Título conclusivo (escrito antes de dibujar, se valida en el paso c):**
*"La forma del rendimiento del plátano es estable año a año: mediana cerca de 7 t/ha y una cola alta
hasta 25-30, con un leve repunte en 2025."*

**Paso (b): figura interactiva + contraste.** Plotly no trae el Ridge como función: se construye a
mano con una KDE por año sobre un **eje x común** ([0, 30] t/ha), desplazadas verticalmente y
**sin extrapolar** (cada densidad se evalúa solo dentro del rango observado de su año). Al lado, el
`go.Box` del mismo corte. Exporta a HTML autónomo y PNG.

In [ ]:
from scipy.stats import gaussian_kde
from plotly.subplots import make_subplots

TITULO_F2 = ("Plátano: la forma del rendimiento es estable 2019-2025 — mediana cerca de 7 t/ha "
             "y cola alta hasta 25-30, con leve repunte en 2025")

orden_anios = tabla_platano.index.tolist()          # ascendente: 2019 .. 2025
etiquetas_anio = [f"{int(a)}  (n={int(tabla_platano.loc[a, 'N'])})" for a in orden_anios]
H_ridge = 0.95

fig2 = make_subplots(rows=1, cols=2, shared_yaxes=False, column_widths=[0.62, 0.38],
                     horizontal_spacing=0.06,
                     subplot_titles=("Ridge · densidad KDE por año (eje x común)",
                                     "Boxplot tradicional (mismo corte)"))

for i, anio in enumerate(orden_anios):
    sub = platano[platano["anio"] == anio]
    v = sub["rendimiento"].to_numpy()
    grid = np.linspace(v.min(), v.max(), 250)          # sin extrapolar: solo rango observado
    dens = gaussian_kde(v)(grid)
    alt = dens / dens.max() * H_ridge
    med = float(np.median(v))
    q17, q83 = np.quantile(v, [0.17, 0.83])
    # densidad (cresta)
    fig2.add_trace(go.Scatter(
        x=np.concatenate([grid, grid[::-1]]),
        y=np.concatenate([i + alt, np.full_like(grid, i)]),
        fill="toself", fillcolor="rgba(150,150,150,0.35)", line=dict(color=ACENTO, width=1.2),
        hoverinfo="text", showlegend=False,
        hovertext=f"{int(anio)}<br>n = {len(v)}<br>mediana = {med:.1f} t/ha<br>IQR 66% = {q17:.1f}-{q83:.1f}"),
        row=1, col=1)
    # mediana (linea vertical)
    fig2.add_trace(go.Scatter(x=[med, med], y=[i, i + H_ridge * 0.9], mode="lines",
        line=dict(color=ACENTO, width=2), hoverinfo="text", showlegend=False,
        hovertext=f"{int(anio)} · mediana {med:.1f} t/ha"), row=1, col=1)
    # puntos crudos con jitter fijo (hover por registro)
    rng = np.random.default_rng(20261014)
    jit = rng.uniform(0, 0.12, len(v))
    hov = [f"Municipio: {mu}<br>Departamento: {de}<br>Cultivo: {cu}<br>Periodo: {pe}"
           f"<br>Rendimiento: {r:.1f} t/ha<br>n del anio: {len(v)}"
           for mu, de, cu, pe, r in zip(sub["municipio"], sub["departamento"],
                                        sub["cultivo"], sub["periodo"], v)]
    fig2.add_trace(go.Scatter(x=v, y=i + jit, mode="markers",
        marker=dict(color=ACENTO, size=3, opacity=0.22), hoverinfo="text",
        hovertext=hov, showlegend=False), row=1, col=1)

# Boxplot de contraste
for anio in orden_anios:
    v = platano.loc[platano["anio"] == anio, "rendimiento"]
    fig2.add_trace(go.Box(x=v, name=str(int(anio)), orientation="h",
        marker_color=GRISES[1], fillcolor="rgba(150,150,150,0.35)",
        line=dict(color=GRISES[1]), boxpoints="outliers", marker=dict(color=GRISES[1], size=3),
        showlegend=False, hovertemplate=f"{int(anio)}<br>%{{x:.1f}} t/ha<extra></extra>"), row=1, col=2)

fig2.update_yaxes(tickmode="array", tickvals=list(range(len(orden_anios))), ticktext=etiquetas_anio,
                  range=[-0.3, len(orden_anios) - 1 + H_ridge + 0.3], row=1, col=1)
fig2.update_yaxes(type="category", categoryorder="array", categoryarray=[str(int(a)) for a in orden_anios],
                  tickmode="array", tickvals=[str(int(a)) for a in orden_anios], ticktext=etiquetas_anio,
                  row=1, col=2)
fig2.update_xaxes(title_text="Rendimiento (t/ha) - escala lineal", range=[0, 30], row=1, col=1)
fig2.update_xaxes(title_text="Rendimiento (t/ha) - escala lineal", range=[0, 30], row=1, col=2)
fig2.update_layout(title=dict(text=TITULO_F2, font=dict(size=15)), width=1080, height=560,
    margin=dict(l=10, r=10, t=90, b=70), plot_bgcolor="white", paper_bgcolor="white")
fuente_ann2 = dict(text=FUENTE_FIG, showarrow=False, xref="paper", yref="paper",
                   x=0, y=-0.13, xanchor="left", font=dict(size=10, color=TEXTO))
fig2.update_layout(annotations=list(fig2.layout.annotations) + [fuente_ann2])

exportar_figura(fig2, "fig2_ridge_platano", 1080, 560)
print("Traces en la figura:", len(fig2.data))
print("Etiquetas de n por anio:", etiquetas_anio)

fig2.show()   # dibuja la figura dentro del cuaderno (como plt.show())

**Paso (c): interpretación y validación del título.** Se calculan los estadísticos por año y el
párrafo se arma con *f-strings*. Después se **valida el título** contra la tabla (mediana estable,
cola alta y repunte de 2025): si no se sostiene, se reporta en lugar de retocarlo.

In [ ]:
# === Figura 2 · Ridge — Paso (c): interpretacion desde variables ===
resumen_anio = platano.groupby("anio")["rendimiento"].agg(
    n="size", minimo="min",
    p10=lambda s: s.quantile(0.10), p25=lambda s: s.quantile(0.25),
    mediana="median", media="mean",
    p75=lambda s: s.quantile(0.75), p90=lambda s: s.quantile(0.90),
    p95=lambda s: s.quantile(0.95), maximo="max")
resumen_anio["iqr"] = resumen_anio["p75"] - resumen_anio["p25"]
resumen_anio["ancho_p10_p90"] = resumen_anio["p90"] - resumen_anio["p10"]
print(resumen_anio.round(2).to_string())

med_min, med_max = resumen_anio["mediana"].min(), resumen_anio["mediana"].max()
rango_medianas = med_max - med_min
iqr_min, iqr_max = resumen_anio["iqr"].min(), resumen_anio["iqr"].max()
cola_min = resumen_anio["maximo"].min()
anio_2025 = int(resumen_anio.index.max())
med_2025 = float(resumen_anio.loc[anio_2025, "mediana"])
med_prev_max = float(resumen_anio.loc[resumen_anio.index < anio_2025, "mediana"].max())
anios_media_supera = int((resumen_anio["media"] > resumen_anio["mediana"]).sum())

texto_f2 = (
    f"En platano, la mediana del rendimiento municipal se mantiene practicamente plana entre "
    f"{med_min:.1f} y {med_max:.1f} t/ha en los siete anios (rango de medianas de solo "
    f"{rango_medianas:.1f} t/ha). La dispersion tampoco cambia de forma apreciable: el rango "
    f"intercuartilico se queda entre {iqr_min:.1f} y {iqr_max:.1f} t/ha. El unico movimiento "
    f"visible es un leve repunte en {anio_2025} ({med_2025:.2f} t/ha frente al maximo previo de "
    f"{med_prev_max:.2f}). El maximo anual supera en todos los anios las {cola_min:.0f} t/ha: hay "
    f"una cola alta persistente de municipios muy por encima de la mediana. La media supera a la "
    f"mediana en {anios_media_supera} de {len(resumen_anio)} anios, es decir la asimetria se "
    f"repite. En conjunto, entre anios cambia poco y cambia sobre todo de mediana, no de forma. "
    f"El analisis es descriptivo: no se atribuyen causas a los movimientos."
)
INTERPRETACIONES["fig2_ridge"] = texto_f2
print(texto_f2)

print("\n--- Validacion del titulo conclusivo ---")
v_med = bool(resumen_anio["mediana"].between(6.5, 8.0).all())
v_cola = bool((resumen_anio["maximo"] >= 25).all())
v_repunte = bool(med_2025 > med_prev_max)
print("  'mediana cerca de 7 t/ha'      :", v_med, "->", list(resumen_anio["mediana"].round(2)))
print("  'cola alta hasta 25-30'        :", v_cola, "-> max por anio:", list(resumen_anio["maximo"].round(1)))
print(f"  'leve repunte en {anio_2025}'          :", v_repunte, f"-> {med_2025:.2f} vs max previo {med_prev_max:.2f}")
print("  Titulo sostenido por la tabla:", bool(v_med and v_cola and v_repunte))

(DIR_HTML.parent / "interpretaciones.json").write_text(
    json.dumps(INTERPRETACIONES, ensure_ascii=False, indent=2), encoding="utf-8")
print("Interpretaciones guardadas en interpretaciones.json")

## 4 · Figura 3 — Sinaplot · cacao por departamento

**Concepto.** El *Sinaplot* es un gráfico de puntos en el que cada observación se coloca sobre la
categoría, con un desplazamiento horizontal proporcional a la **densidad local** (estimada con una
KDE). Combina la lectura de un *strip/violín* (se ven **todos los puntos**, no solo la caja) con la
forma de la distribución, como un violín delgado.

**Uso recomendado.** Mostrar distribuciones con **N moderado** donde importa ver cada dato y su
densidad a la vez (mejor que la caja sola y más informativo que el violín, que no muestra puntos).
**No** conviene con N muy grande (satura) ni con pocos puntos por grupo (el jitter hacia la densidad
pierde sentido).

**Paso (a): tabla del corte.** Se filtra cacao (cultivo permanente → convención anual) en seis
departamentos, se descartan los registros sin cosecha y se tabula N por departamento.

In [ ]:
print("=== Figura 3 - Sinaplot: cacao por departamento (convencion anual) ===")
print("Pregunta: como se distribuye el rendimiento del cacao entre municipios en Huila, Tolima, Valle")
print("del Cauca, Cundinamarca, Cauca y Meta, y la mediana y la cola distinguen claramente a alguno?")

DEC3 = ["Huila", "Tolima", "Valle del Cauca", "Cundinamarca", "Cauca", "Meta"]
m1 = df["cultivo"] == "Cacao"
print(f"Filtro 1 cultivo=='Cacao'            : conserva {int(m1.sum()):6d} de {len(df):6d} (quita {int((~m1).sum()):6d})")
paso3 = df[m1]
m2 = paso3["rendimiento"] > 0
print(f"Filtro 2 rendimiento>0               : conserva {int(m2.sum()):6d} de {len(paso3):6d} (quita {int((~m2).sum()):6d})")
paso3 = paso3[m2]
m3 = paso3["departamento"].isin(DEC3)
print(f"Filtro 3 depto en los 6 seleccionados: conserva {int(m3.sum()):6d} de {len(paso3):6d} (quita {int((~m3).sum()):6d})")
cacao = paso3[m3].copy()

tabla_cacao = (cacao.groupby("departamento")
               .agg(N=("rendimiento", "size"),
                    municipios=("codigo_dane_municipio", "nunique"),
                    mediana=("rendimiento", "median"),
                    p25=("rendimiento", lambda s: s.quantile(0.25)),
                    p75=("rendimiento", lambda s: s.quantile(0.75)),
                    maximo=("rendimiento", "max"))
               .sort_values("mediana"))
print("\nTabla del corte (Sinaplot), por departamento (ordenada por mediana):")
print(tabla_cacao.round(2).to_string())

ymin_ca, ymax_ca = float(cacao["rendimiento"].min()), float(cacao["rendimiento"].max())
print(f"\nRango y de los puntos: [{ymin_ca:.1f}, {ymax_ca:.1f}] t/ha")
print(f"Comprobacion: deptos = {tabla_cacao.index.tolist()} | N minimo = {int(tabla_cacao['N'].min())} "
      f"| deptos con N>0 = {int((tabla_cacao['N'] > 0).sum())} de {len(tabla_cacao)} "
      f"| convencion de periodo = {cacao['tipo_periodo'].unique().tolist()}")

**Pregunta (Figura 3):** ¿cómo se distribuye el rendimiento del cacao entre municipios en esos seis
departamentos, y la mediana o la cola distinguen claramente a alguno?

**Título conclusivo (escrito antes de dibujar, se valida en el paso c):**
*"Cacao: medianas casi iguales (0,5-0,6 t/ha) pero colas distintas - Valle del Cauca y Cauca llegan
a 1,4-1,5 t/ha, mientras Huila se queda en 0,9."*

**Paso (b): figura interactiva + contraste.** Sinaplot construido a mano: cada municipio es un punto
sobre la columna de su departamento, desplazado horizontalmente en proporción a la **densidad local**
(KDE). Eje y común con el `go.Box` del mismo corte a la derecha. Exporta a HTML y PNG.

In [ ]:
from scipy.stats import gaussian_kde

TITULO_F3 = ("Cacao: medianas casi iguales (0,5-0,6 t/ha) pero colas distintas - "
             "Valle del Cauca y Cauca llegan a 1,4-1,5 t/ha; Huila se queda en 0,9")

deptos_orden = tabla_cacao.index.tolist()            # ordenados por mediana (ascendente)
W_SINA = 0.38                                        # semiancho maximo (en unidades de categoria)
rng3 = np.random.default_rng(20261014)

fig3 = make_subplots(rows=1, cols=2, shared_yaxes=True, column_widths=[0.6, 0.4],
                     horizontal_spacing=0.05,
                     subplot_titles=("Sinaplot · jitter proporcional a la densidad por departamento",
                                     "Boxplot tradicional (mismo corte)"))

for j, dep in enumerate(deptos_orden):
    sub = cacao[cacao["departamento"] == dep]
    v = sub["rendimiento"].to_numpy()
    dens = gaussian_kde(v)(v)
    semiancho = dens / dens.max() * W_SINA
    off = rng3.uniform(-1, 1, len(v)) * semiancho
    n_dep = len(v)
    hov = [f"Municipio: {mu}<br>Departamento: {de}<br>Cultivo: {cu}<br>Periodo: {pe}"
           f"<br>Rendimiento: {r:.2f} t/ha<br>n del depto: {n_dep}"
           for mu, de, cu, pe, r in zip(sub["municipio"], sub["departamento"],
                                        sub["cultivo"], sub["periodo"], v)]
    fig3.add_trace(go.Scatter(x=j + off, y=v, mode="markers",
        marker=dict(color=ACENTO, size=5, opacity=0.45, line=dict(width=0)),
        hoverinfo="text", hovertext=hov, showlegend=False), row=1, col=1)
    fig3.add_annotation(x=j, y=1.58, text=f"n={n_dep}", showarrow=False,
                        font=dict(size=11, color=TEXTO), row=1, col=1)

for dep in deptos_orden:
    v = cacao.loc[cacao["departamento"] == dep, "rendimiento"]
    fig3.add_trace(go.Box(y=v, name=dep, marker_color=GRISES[1],
        fillcolor="rgba(150,150,150,0.35)", line=dict(color=GRISES[1]),
        boxpoints="all", jitter=0.3, pointpos=0, marker=dict(color=GRISES[1], size=3),
        showlegend=False, hovertemplate=f"{dep}<br>%{{y:.2f}} t/ha<extra></extra>"), row=1, col=2)

fig3.update_xaxes(tickmode="array", tickvals=list(range(len(deptos_orden))), ticktext=deptos_orden,
                  tickangle=0, title_text="Departamento (ordenado por mediana)", row=1, col=1)
# El panel de contraste tiene menos ancho: las etiquetas se rotan para que no se solapen.
fig3.update_xaxes(type="category", categoryorder="array", categoryarray=deptos_orden,
                  ticktext=deptos_orden, tickangle=-35, tickfont=dict(size=10),
                  title_text="Departamento (ordenado por mediana)",
                  row=1, col=2)
fig3.update_yaxes(title_text="Rendimiento (t/ha) - escala lineal", range=[0, 1.7], row=1, col=1)
fig3.update_layout(title=dict(text=TITULO_F3, font=dict(size=15)), width=1240, height=640,
    margin=dict(l=30, r=30, t=95, b=135), plot_bgcolor="white", paper_bgcolor="white")
fuente_ann3 = dict(text=FUENTE_FIG, showarrow=False, xref="paper", yref="paper",
                   x=0, y=-0.24, xanchor="left", font=dict(size=10, color=TEXTO))
fig3.update_layout(annotations=list(fig3.layout.annotations) + [fuente_ann3])

exportar_figura(fig3, "fig3_sinaplot_cacao", 1240, 640)
print("Traces en la figura:", len(fig3.data))
print("n por depto (anotaciones):", [f"{d}={int(tabla_cacao.loc[d, 'N'])}" for d in deptos_orden])

fig3.show()   # dibuja la figura dentro del cuaderno (como plt.show())

**Paso (c): interpretación y validación del título.** Se calculan los estadísticos por departamento y
el párrafo se arma con *f-strings*. Después se **valida el título** (medianas casi iguales y colas
distintas): si no se sostiene, se reporta en lugar de retocarlo.

In [ ]:
# === Figura 3 · Sinaplot — Paso (c): interpretacion desde variables ===
resumen_ca = cacao.groupby("departamento")["rendimiento"].agg(
    n="size", minimo="min",
    p25=lambda s: s.quantile(0.25), mediana="median", media="mean",
    p75=lambda s: s.quantile(0.75), p90=lambda s: s.quantile(0.90),
    p95=lambda s: s.quantile(0.95), maximo="max")
resumen_ca["iqr"] = resumen_ca["p75"] - resumen_ca["p25"]
print(resumen_ca.round(3).to_string())

med_min, med_max = resumen_ca["mediana"].min(), resumen_ca["mediana"].max()
iqr_min, iqr_max = resumen_ca["iqr"].min(), resumen_ca["iqr"].max()
top_cola = resumen_ca["maximo"].sort_values(ascending=False)
d_cola1, d_cola2 = top_cola.index[0], top_cola.index[1]
m_cola1, m_cola2 = float(top_cola.iloc[0]), float(top_cola.iloc[1])
d_baja = resumen_ca["maximo"].idxmin()
m_baja = float(resumen_ca["maximo"].min())
skew_n = int((resumen_ca["media"] > resumen_ca["mediana"]).sum())

texto_f3 = (
    f"En cacao, las medianas departamentales son casi indistinguibles: van de {med_min:.2f} a "
    f"{med_max:.2f} t/ha (una diferencia de apenas {med_max - med_min:.2f}). La separacion aparece "
    f"en la cola alta: {d_cola1} y {d_cola2} alcanzan maximos de {m_cola1:.2f} y {m_cola2:.2f} "
    f"t/ha, mientras {d_baja} se queda en {m_baja:.2f}. El rango intercuartilico es estrecho en "
    f"todos (entre {iqr_min:.2f} y {iqr_max:.2f} t/ha): el municipio tipico produce alrededor de "
    f"medio kilo por hectarea, y unos pocos municipios duplican o triplican la mediana. La media "
    f"supera a la mediana en {skew_n} de {len(resumen_ca)} departamentos: la misma asimetria "
    f"positiva. El analisis es descriptivo: no se atribuyen causas a las diferencias."
)
INTERPRETACIONES["fig3_sinaplot"] = texto_f3
print(texto_f3)

print("\n--- Validacion del titulo conclusivo (escrito antes de dibujar) ---")
v_medianas = bool(resumen_ca["mediana"].between(0.45, 0.65).all())
v_cola = bool(set(top_cola.index[:2]) == {"Valle del Cauca", "Cauca"} and m_cola2 >= 1.4)
v_baja = bool(resumen_ca.loc["Huila", "maximo"] < 1.0)
print("  'medianas casi iguales (0,5-0,6)'    :", v_medianas, "->", list(resumen_ca["mediana"].round(2)))
print("  'colas distintas (Valle/Cauca 1,4-1,5)':", v_cola, f"-> top: {d_cola1}={m_cola1:.2f}, {d_cola2}={m_cola2:.2f}")
print("  'Huila se queda en 0,9'              :", v_baja, f"-> Huila max {resumen_ca.loc['Huila','maximo']:.2f}")
print("  Titulo sostenido por la tabla:", bool(v_medianas and v_cola and v_baja))

(DIR_HTML.parent / "interpretaciones.json").write_text(
    json.dumps(INTERPRETACIONES, ensure_ascii=False, indent=2), encoding="utf-8")
print("Interpretaciones guardadas en interpretaciones.json")

## 5 · Figura 4 — Boxenplot (Letter-Value Plot) · cinco cultivos con más registros, rendimiento relativo

**Concepto.** El *Boxenplot* (o *letter-value plot*) dibuja **muchos más cuantiles** que la caja: un
conjunto de bandas anidadas que se estrechan hacia las colas. Su grosor visual es predefinido; la
información estadística está en los extremos de cuantiles de cada banda. Corrige la limitación del boxplot de resumir toda la distribución en cinco
números y perder el detalle de las colas, sobre todo cuando N es grande y hay observaciones de sobra
para estimar cuantiles extremos.

**Uso recomendado.** Distribuciones con **N grande** (miles de observaciones) donde importa la
**cola**; en muestras pequeñas no hay datos suficientes para tantos cuantiles y conviene la caja.
Aquí se comparan **cinco cultivos** usando el **rendimiento relativo** (rendimiento ÷ mediana nacional
del cultivo), porque los rendimientos absolutos no son comparables entre cultivos.

**Paso (a): tabla del corte.** Se toman los cinco cultivos con más registros, se descartan los
registros sin cosecha y se normaliza por la mediana nacional de cada cultivo.

In [ ]:
print("=== Figura 4 - Boxenplot: cinco cultivos con mas registros, rendimiento relativo ===")
print("Pregunta: comparados en una escala comun (rendimiento / mediana nacional), que cultivo tiene")
print("la cola mas pesada y cual la cola mas liviana entre los cinco con mas registros?")

CULT5 = ["Maíz", "Yuca", "Frijol", "Tomate", "Plátano"]
top5_calculado = (df.loc[df["rendimiento"] > 0, "cultivo"]
                  .value_counts().head(5).index.tolist())
assert CULT5 == top5_calculado, "La selección debe coincidir con los cinco mayores N del corte EVA"
m1 = df["cultivo"].isin(CULT5)
print(f"Filtro 1 cinco mayores N EVA    : conserva {int(m1.sum()):6d} de {len(df):6d} (quita {int((~m1).sum()):6d})")
paso4 = df[m1]
m2 = paso4["rendimiento"] > 0
print(f"Filtro 2 rendimiento>0         : conserva {int(m2.sum()):6d} de {len(paso4):6d} (quita {int((~m2).sum()):6d})")
top5 = paso4[m2].copy()

med_nac = top5.groupby("cultivo")["rendimiento"].median()
top5["rend_rel"] = top5["rendimiento"] / top5["cultivo"].map(med_nac)

tabla5 = (top5.groupby("cultivo")
          .agg(N=("rend_rel", "size"),
               municipios=("codigo_dane_municipio", "nunique"),
               mediana_nacional=("rendimiento", "median"),
               rel_p25=("rend_rel", lambda s: s.quantile(0.25)),
               rel_mediana=("rend_rel", "median"),
               rel_p75=("rend_rel", lambda s: s.quantile(0.75)),
               rel_p99=("rend_rel", lambda s: s.quantile(0.99)))
          .sort_values("rel_mediana"))
print("\nTabla del corte (Boxenplot), por cultivo:")
print(tabla5.round(3).to_string())

print(f"\nRendimiento relativo minimo = {top5['rend_rel'].min():.3f} | maximo = {top5['rend_rel'].max():.3f}")
print(f"Comprobacion: cultivos = {tabla5.index.tolist()} | N minimo = {int(tabla5['N'].min())} "
      f"| cultivos con N>0 = {int((tabla5['N'] > 0).sum())} de {len(CULT5)} "
      f"| medianas relativas en [0.99, 1.01] = {bool(tabla5['rel_mediana'].between(0.99, 1.01).all())}")

**Pregunta (Figura 4):** comparados en una escala común (rendimiento ÷ mediana nacional), ¿qué cultivo
tiene la cola más pesada y cuál la más liviana entre los cinco más sembrados?

**Título conclusivo (escrito antes de dibujar, se valida en el paso c):**
*"El nivel es el mismo (medianas relativas = 1) pero las colas no: tomate y maíz tienen la cola alta
más pesada (p99 ≈ 4,4-4,8) y la yuca la más liviana (p99 ≈ 2,5)."*

**Paso (b): figura interactiva + contraste.** Plotly no trae el *letter-value plot*: se construye a
mano con **cajas anidadas** de cuantiles cada vez más extremos (cuartos, octavos, dieciseisavos,
treintaidosavos, sesentaicuatroavos) y la mediana marcada. La variable es un **cociente**, así que se
usa **escala logarítmica** (declarada) y una referencia en 1 (la mediana nacional). Al lado, el
`go.Box` del mismo corte. Los puntos del hover son una **muestra** por cultivo (se declara).

**Nota de orden (encargo v4):** las medianas relativas empatan (=1,00) en los cinco cultivos, así que
"ordenar por mediana" no discrimina. Se desempata por el **cuantil de cola p99** (ascendente), que es
lo que el Boxenplot mide, y se **declara en el eje**.

In [ ]:
TITULO_F4 = ("Cinco cultivos con más registros EVA: mismo nivel (medianas relativas = 1), pero colas distintas<br>"
             "tomate y maíz con la cola alta más pesada (p99 4,4–4,8), yuca con la más liviana (p99 2,5)")

cult5_orden = tabla5.sort_values(["rel_mediana", "rel_p99"]).index.tolist()  # empate de mediana -> desempate por p99 (v4)
niveles = [(0.25, 0.75), (0.125, 0.875), (0.0625, 0.9375), (0.03125, 0.96875), (0.015625, 0.984375)]
grosores = [0.90, 0.52, 0.30, 0.17, 0.09]
etiquetas5 = [f"{c}  (n={int(tabla5.loc[c, 'N'])})" for c in cult5_orden]
rng4 = np.random.default_rng(20261014)
MUESTRA_RUG = 1500

fig4 = make_subplots(rows=1, cols=2, shared_yaxes=False, column_widths=[0.64, 0.36],
                     horizontal_spacing=0.06,
                     subplot_titles=("Boxenplot · letter-value (cuantiles anidados)",
                                     "Boxplot tradicional (mismo corte)"))

for i, c in enumerate(cult5_orden):
    v = top5.loc[top5["cultivo"] == c, "rend_rel"].to_numpy()
    # cajas anidadas: de la mas profunda (ancha) a la mas somera (gruesa), para que el centro quede arriba
    for nivel, (qlo, qhi) in enumerate(reversed(niveles)):
        x0, x1 = np.quantile(v, [qlo, qhi])
        g = grosores[len(niveles) - 1 - nivel] / 2
        fig4.add_trace(go.Scatter(
            x=[x0, x1, x1, x0, x0], y=[i - g, i - g, i + g, i + g, i - g],
            fill="toself", fillcolor="rgba(150,150,150,0.28)", line=dict(color=ACENTO, width=1.1),
            hoverinfo="text", showlegend=False,
            hovertext=f"{c}<br>cuantiles {qlo:.4f}-{qhi:.4f}<br>{x0:.2f} - {x1:.2f} x mediana nacional"),
            row=1, col=1)
    # mediana
    med = float(np.median(v))
    fig4.add_trace(go.Scatter(x=[med, med], y=[i - 0.47, i + 0.47], mode="lines",
        line=dict(color=ACENTO, width=3), hoverinfo="text", showlegend=False,
        hovertext=f"{c} · mediana relativa {med:.2f}"), row=1, col=1)
    # capa de puntos (muestra) con hover por registro
    sel = top5[top5["cultivo"] == c]
    idx = rng4.choice(len(sel), size=min(MUESTRA_RUG, len(sel)), replace=False)
    muestra = sel.iloc[idx]
    jit = rng4.uniform(-0.42, 0.42, len(muestra))
    hov = [f"Municipio: {mu}<br>Departamento: {de}<br>Cultivo: {cu}<br>Periodo: {pe}"
           f"<br>Rendimiento: {rr:.2f} t/ha<br>Relativo: {rel:.2f} x<br>n del cultivo: {int(tabla5.loc[c, 'N'])}"
           for mu, de, cu, pe, rr, rel in zip(muestra["municipio"], muestra["departamento"],
                                              muestra["cultivo"], muestra["periodo"],
                                              muestra["rendimiento"], muestra["rend_rel"])]
    fig4.add_trace(go.Scatter(x=muestra["rend_rel"], y=i + jit, mode="markers",
        marker=dict(color=ACENTO, size=3, opacity=0.18), hoverinfo="text",
        hovertext=hov, showlegend=False), row=1, col=1)

for c in cult5_orden:
    v = top5.loc[top5["cultivo"] == c, "rend_rel"]
    fig4.add_trace(go.Box(x=v, name=c, orientation="h",
        marker_color=GRISES[1], fillcolor="rgba(150,150,150,0.35)", line=dict(color=GRISES[1]),
        boxpoints="outliers", marker=dict(color=GRISES[1], size=3), showlegend=False,
        hovertemplate=f"{c}<br>%{{x:.2f}} x mediana<extra></extra>"), row=1, col=2)

fig4.add_vline(x=1, line=dict(color=GRISES[0], width=1.5, dash="dot"), row=1, col=1)
fig4.add_annotation(x=1, y=len(cult5_orden) - 0.5, text="mediana nacional = 1", showarrow=False,
                    font=dict(size=10, color=GRISES[0]), row=1, col=1)

fig4.update_yaxes(tickmode="array", tickvals=list(range(len(cult5_orden))), ticktext=etiquetas5,
                  title_text="Cultivo (orden por cola p99; medianas relativas empatan)",
                  range=[-0.7, len(cult5_orden) - 0.3], row=1, col=1)
fig4.update_yaxes(type="category", categoryorder="array", categoryarray=cult5_orden,
                  title_text="Cultivo (orden por cola p99; medianas relativas empatan)",
                  tickmode="array", tickvals=cult5_orden, ticktext=etiquetas5, row=1, col=2)
# Referencias limitadas y explicitas: una escala logaritmica debe poder leerse sin adivinar exponentes.
marcas_log = [0.01, 0.1, 0.5, 1, 2, 5, 10]
etiquetas_log = ["0,01×", "0,1×", "0,5×", "1×", "2×", "5×", "10×"]
fig4.update_xaxes(type="log", title_text="Rendimiento relativo (× mediana nacional) — escala logarítmica",
                  range=[-2.1, 1.35], tickmode="array", tickvals=marcas_log, ticktext=etiquetas_log, row=1, col=1)
fig4.update_xaxes(type="log", title_text="Rendimiento relativo (× mediana nacional) — escala logarítmica",
                  range=[-2.1, 1.35], tickmode="array", tickvals=marcas_log, ticktext=etiquetas_log, row=1, col=2)
fig4.update_layout(title=dict(text=TITULO_F4, font=dict(size=15)), width=1320, height=650,
    margin=dict(l=45, r=45, t=110, b=120), plot_bgcolor="white", paper_bgcolor="white")
fuente_ann4 = dict(text=FUENTE_FIG + " Capa de puntos: muestra de hasta 1500 por cultivo.",
                   showarrow=False, xref="paper", yref="paper",
                   x=0, y=-0.25, xanchor="left", font=dict(size=10, color=TEXTO))
fig4.update_layout(annotations=list(fig4.layout.annotations) + [fuente_ann4])

exportar_figura(fig4, "fig4_boxenplot_top5", 1320, 650)
print("Traces en la figura:", len(fig4.data))
print("Orden dibujado (medianas empatan -> desempate por p99 ascendente):", cult5_orden)

fig4.show()   # dibuja la figura dentro del cuaderno (como plt.show())

**Paso (c): interpretación y validación del título.** Se calculan los cuantiles relativos por cultivo
y el párrafo se arma con *f-strings*. Después se **valida el título** (nivel igual y colas distintas):
si no se sostiene, se reporta en lugar de retocarlo.

In [ ]:
# === Figura 4 · Boxenplot — Paso (c): interpretacion desde variables ===
resumen_rel = top5.groupby("cultivo")["rend_rel"].agg(
    n="size", rel_p10=lambda s: s.quantile(0.10),
    rel_p25=lambda s: s.quantile(0.25), rel_mediana="median",
    rel_p75=lambda s: s.quantile(0.75), rel_p90=lambda s: s.quantile(0.90),
    rel_p95=lambda s: s.quantile(0.95), rel_p99=lambda s: s.quantile(0.99),
    rel_max="max")
resumen_rel["iqr_rel"] = resumen_rel["rel_p75"] - resumen_rel["rel_p25"]
resumen_rel["cola_alta_p99_sobre_mediana"] = resumen_rel["rel_p99"] / resumen_rel["rel_mediana"]
print(resumen_rel.round(3).to_string())

p99_orden = resumen_rel["rel_p99"].sort_values(ascending=False)
c_cola_pesada, c_cola_liviana = p99_orden.index[0], p99_orden.index[-1]
p99_pesada, p99_liviana = float(p99_orden.iloc[0]), float(p99_orden.iloc[-1])
p99_tomate = float(resumen_rel.loc["Tomate", "rel_p99"])
p99_maiz = float(resumen_rel.loc["Maíz", "rel_p99"])
p25_tomate = float(resumen_rel.loc["Tomate", "rel_p25"])
p25_yuca = float(resumen_rel.loc["Yuca", "rel_p25"])
max_tomate = float(resumen_rel.loc["Tomate", "rel_max"])
c_max = resumen_rel["rel_max"].idxmax()
max_global = float(resumen_rel["rel_max"].max())

texto_f4 = (
    f"Normalizado por la mediana nacional, los cinco cultivos parten del mismo nivel: la mediana "
    f"relativa vale 1,00 en todos. La diferencia esta en la dispersion multiplicativa. La cola alta "
    f"mas pesada es la del tomate (p99 = {p99_tomate:.1f}x la mediana) seguida del maiz "
    f"({p99_maiz:.1f}x), y la mas liviana la de la {c_cola_liviana.lower()} "
    f"(p99 = {p99_liviana:.1f}x). En el lado bajo pasa lo mismo: el tomate es el que mas desciende "
    f"(p25 = {p25_tomate:.2f}x) y la yuca el que menos (p25 = {p25_yuca:.2f}x). En el Boxenplot esto "
    f"se ve como cajas anidadas mas largas en tomate y maiz, y mas cortas en yuca. Lectura practica: "
    f"un municipio con tomate puede rendir hasta {max_tomate:.0f} veces la mediana nacional "
    f"(extremo global del corte: {max_global:.1f}x en {c_max}); la caja tradicional (p25-p75) apenas "
    f"insinuaria esa cola. El analisis es descriptivo: no se atribuyen causas."
)
INTERPRETACIONES["fig4_boxenplot"] = texto_f4
print(texto_f4)

print("\n--- Validacion del titulo conclusivo (escrito antes de dibujar) ---")
v_nivel = bool(resumen_rel["rel_mediana"].between(0.99, 1.01).all())
v_tm = bool(4.4 <= resumen_rel.loc["Tomate", "rel_p99"] <= 4.9
            and 4.4 <= resumen_rel.loc["Maíz", "rel_p99"] <= 4.9)
v_yuca = bool(resumen_rel["rel_p99"].idxmin() == "Yuca")
print("  'nivel igual (medianas = 1)'        :", v_nivel, "->", list(resumen_rel["rel_mediana"].round(2)))
print("  'tomate y maiz con la cola p99 4,4-4,8':", v_tm,
      f"-> tomate {resumen_rel.loc['Tomate','rel_p99']:.2f}, maiz {resumen_rel.loc['Maíz','rel_p99']:.2f}")
print("  'yuca con la cola mas liviana'      :", v_yuca, "-> p99 yuca", f"{resumen_rel.loc['Yuca','rel_p99']:.2f}")
print("  Titulo sostenido por la tabla:", bool(v_nivel and v_tm and v_yuca))

(DIR_HTML.parent / "interpretaciones.json").write_text(
    json.dumps(INTERPRETACIONES, ensure_ascii=False, indent=2), encoding="utf-8")
print("Interpretaciones guardadas en interpretaciones.json")

## 6 · Figura 5 — Composite Stat-Boxplot · Café × 6 departamentos

**Concepto.** El *Composite Stat-Boxplot* no es una técnica de dibujo distinta, sino la **síntesis**
de todas las anteriores en una sola vista: superpone la **caja letter-value** (cuantiles anidados, del
Boxenplot), la **mediana**, la **media** (clave cuando el sesgo se invierte) y una **capa de puntos**
individuales (de la familia Sinaplot/jitter). Corrige la limitación de elegir *una* vista: permite leer
de un golpe forma, centro, dispersión, cola y valor atípico, sin cambiar de gráfico.

**Uso recomendado.** Cuando hay **varias cosas que decir a la vez** y el público no va a interactuar
con varias figuras; en audiencia general conviene no sobrecargar y reservarlo para el caso que lo
merezca. Aquí se compara el **rendimiento del café** (t/ha, café pergamino seco) en los seis
departamentos cafeteros más representativos de los datos.

**Paso (a): tabla del corte.** Se toman los registros de Café en los seis departamentos y se descartan
los que no tuvieron producción.

In [ ]:
print("=== Figura 5 - Composite Stat-Boxplot: Cafe x 6 departamentos ===")
print("Pregunta: entre los seis departamentos cafeteros, donde rinde mas el cafe por hectarea y")
print("como cambia el centro (mediana vs media) al mirar los extremos?")

DEP6 = ["Antioquia", "Santander", "Cundinamarca", "Boyacá", "Nariño", "Valle del Cauca"]
m0 = df["cultivo"] == "Café"
print(f"Filtro 0 cultivo == Cafe       : conserva {int(m0.sum()):6d} de {len(df):6d} (quita {int((~m0).sum()):6d})")
cafe = df[m0]
m1 = cafe["departamento"].isin(DEP6)
print(f"Filtro 1 deptos cafe           : conserva {int(m1.sum()):6d} de {len(cafe):6d} (quita {int((~m1).sum()):6d})")
paso6 = cafe[m1]
m2 = paso6["rendimiento"] > 0
print(f"Filtro 2 rendimiento>0         : conserva {int(m2.sum()):6d} de {len(paso6):6d} (quita {int((~m2).sum()):6d})")
compo = paso6[m2].copy()

tabla6 = (compo.groupby("departamento")["rendimiento"]
          .agg(N="size",
               p25=lambda s: s.quantile(0.25), mediana="median", media="mean",
               p75=lambda s: s.quantile(0.75), p99=lambda s: s.quantile(0.99), maximo="max")
          .sort_values("mediana"))
print("\nTabla del corte (Composite), por departamento:")
print(tabla6.round(3).to_string())

print(f"\nRendimiento minimo = {compo['rendimiento'].min():.2f} | maximo = {compo['rendimiento'].max():.2f} t/ha")
print(f"Comprobacion: deptos = {len(tabla6)} de {len(DEP6)} | N minimo = {int(tabla6['N'].min())} "
      f"| media > mediana en = {int((tabla6['media'] > tabla6['mediana']).sum())} de {len(tabla6)} deptos")

**Pregunta (Figura 5):** entre los seis departamentos cafeteros, ¿dónde rinde más el café por hectárea
y cómo cambia el centro (mediana frente a media) al mirar los extremos?

**Título conclusivo (escrito antes de dibujar, se valida en el paso c):**
*"En los seis departamentos la media es levemente menor que la mediana; las medianas van de 0,95
a 1,21 t/ha."*

**Paso (b): figura interactiva + contraste.** Panel izquierdo: **Composite Stat-Boxplot**, que
superpone —en una sola vista— las cajas letter-value (cuantiles anidados), la mediana, la **media**
(marcador rombo, distinta de la mediana) y una capa de puntos individuales (muestra). Panel derecho: el
`go.Box` clásico del mismo corte. Variable en t/ha de rango estrecho → **escala lineal** (declarada).

**Nota de orden (encargo v4):** departamentos ordenados por mediana ascendente; si dos empatan, se
desempata por el cuantil de cola **p99** (ascendente). Se declara en el eje.

In [ ]:
TITULO_F5 = ("Café: media levemente menor que mediana en los 6 departamentos; medianas de 0,95 a 1,21 t/ha")

dep6_orden = tabla6.sort_values(["mediana", "p99"]).index.tolist()   # v4: mediana; desempate por p99
etiquetas6 = [f"{d}  (n={int(tabla6.loc[d, 'N'])})" for d in dep6_orden]
niveles6 = [(0.25, 0.75), (0.125, 0.875), (0.0625, 0.9375), (0.03125, 0.96875), (0.015625, 0.984375)]
grosores6 = [0.90, 0.52, 0.30, 0.17, 0.09]
rng5 = np.random.default_rng(20261014)
MUESTRA5 = 700

fig5 = make_subplots(rows=1, cols=2, shared_yaxes=False, column_widths=[0.64, 0.36],
                     horizontal_spacing=0.06,
                     subplot_titles=("Composite Stat-Boxplot (letter-value + mediana + media + puntos)",
                                     "Boxplot tradicional (mismo corte)"))

for i, d in enumerate(dep6_orden):
    v = compo.loc[compo["departamento"] == d, "rendimiento"].to_numpy()
    for nivel, (qlo, qhi) in enumerate(reversed(niveles6)):
        x0, x1 = np.quantile(v, [qlo, qhi])
        g = grosores6[len(niveles6) - 1 - nivel] / 2
        fig5.add_trace(go.Scatter(
            x=[x0, x1, x1, x0, x0], y=[i - g, i - g, i + g, i + g, i - g],
            fill="toself", fillcolor="rgba(150,150,150,0.28)", line=dict(color=ACENTO, width=1.1),
            hoverinfo="text", showlegend=False,
            hovertext=f"{d}<br>cuantiles {qlo:.4f}-{qhi:.4f}<br>{x0:.2f} - {x1:.2f} t/ha"),
            row=1, col=1)
    med = float(np.median(v))
    fig5.add_trace(go.Scatter(x=[med, med], y=[i - 0.47, i + 0.47], mode="lines",
        line=dict(color=ACENTO, width=3), hoverinfo="text", showlegend=False,
        hovertext=f"{d} · mediana {med:.2f} t/ha"), row=1, col=1)
    media = float(np.mean(v))
    fig5.add_trace(go.Scatter(x=[media], y=[i], mode="markers",
        marker=dict(symbol="diamond", size=10, color=GRISES[0],
                    line=dict(color="white", width=1)), showlegend=False,
        hoverinfo="text", hovertext=f"{d} · media {media:.2f} t/ha"), row=1, col=1)
    sel = compo[compo["departamento"] == d]
    idx = rng5.choice(len(sel), size=min(MUESTRA5, len(sel)), replace=False)
    muestra = sel.iloc[idx]
    jit = rng5.uniform(-0.38, 0.38, len(muestra))
    hov = [f"Municipio: {mu}<br>Departamento: {de}<br>Cultivo: Café<br>Periodo: {pe}"
           f"<br>Rendimiento: {rr:.2f} t/ha<br>n del departamento: {int(tabla6.loc[d, 'N'])}"
           for mu, de, pe, rr in zip(muestra["municipio"], muestra["departamento"],
                                     muestra["periodo"], muestra["rendimiento"])]
    fig5.add_trace(go.Scatter(x=muestra["rendimiento"], y=i + jit, mode="markers",
        marker=dict(color=ACENTO, size=3, opacity=0.16), hoverinfo="text",
        hovertext=hov, showlegend=False), row=1, col=1)

for d in dep6_orden:
    v = compo.loc[compo["departamento"] == d, "rendimiento"]
    fig5.add_trace(go.Box(x=v, name=d, orientation="h",
        marker_color=GRISES[1], fillcolor="rgba(150,150,150,0.35)", line=dict(color=GRISES[1]),
        boxpoints="outliers", marker=dict(color=GRISES[1], size=3), showlegend=False,
        hovertemplate=f"{d}<br>%{{x:.2f}} t/ha<extra></extra>"), row=1, col=2)

# leyenda explicita del composite (entradas guia)
fig5.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
    marker=dict(symbol="square", size=13, color="rgba(150,150,150,0.6)"),
    name="Caja letter-value (cuantiles)"), row=1, col=1)
fig5.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color=ACENTO, width=3),
    name="Mediana"), row=1, col=1)
fig5.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
    marker=dict(symbol="diamond", size=11, color=GRISES[0]), name="Media"), row=1, col=1)
fig5.add_trace(go.Scatter(x=[None], y=[None], mode="markers",
    marker=dict(symbol="circle", size=6, color=ACENTO), name="Observaciones (muestra)"), row=1, col=1)

fig5.update_yaxes(tickmode="array", tickvals=list(range(len(dep6_orden))), ticktext=etiquetas6,
                  title_text="Departamento (orden por mediana; desempate p99)",
                  range=[-0.7, len(dep6_orden) - 0.3], row=1, col=1)
fig5.update_yaxes(type="category", categoryorder="array", categoryarray=dep6_orden,
                  title_text="Departamento (orden por mediana; desempate p99)",
                  tickmode="array", tickvals=dep6_orden, ticktext=etiquetas6, row=1, col=2)
fig5.update_xaxes(title_text="Rendimiento de café (t/ha) — escala lineal",
                  range=[0, 3.05], row=1, col=1)
fig5.update_xaxes(title_text="Rendimiento de café (t/ha) — escala lineal",
                  range=[0, 3.05], row=1, col=2)
fig5.update_layout(title=dict(text=TITULO_F5, font=dict(size=15)),
    width=1240, height=560, margin=dict(l=10, r=10, t=95, b=80),
    plot_bgcolor="white", paper_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0,
                font=dict(size=10), bgcolor="rgba(0,0,0,0)"))
fuente_ann5 = dict(text=FUENTE_FIG + " Capa de puntos: muestra de hasta 700 por departamento.",
                   showarrow=False, xref="paper", yref="paper",
                   x=0, y=-0.17, xanchor="left", font=dict(size=10, color=TEXTO))
fig5.update_layout(annotations=list(fig5.layout.annotations) + [fuente_ann5])

exportar_figura(fig5, "fig5_composite_cafe", 1240, 560)
print("Traces en la figura:", len(fig5.data))
print("Orden dibujado (mediana; desempate p99):", dep6_orden)

fig5.show()   # dibuja la figura dentro del cuaderno (como plt.show())

**Paso (c): interpretación y validación del título.** Se calculan la brecha media-mediana, el rango
intercuartílico y los extremos por departamento; el párrafo se arma con *f-strings* y se **valida el
título** (media por debajo de la mediana en los 6; rendimientos apretados). Si no se sostiene se
reporta, sin retocarlo.

In [ ]:
# === Figura 5 · Composite — Paso (c): interpretacion desde variables ===
tabla6["brecha_mediana_menos_media"] = tabla6["mediana"] - tabla6["media"]
tabla6["iqr"] = tabla6["p75"] - tabla6["p25"]
print(tabla6.round(3).to_string())

d_gap = tabla6["brecha_mediana_menos_media"].idxmax()
med_gap = float(tabla6.loc[d_gap, "mediana"]); media_gap = float(tabla6.loc[d_gap, "media"])
med_min = float(tabla6["mediana"].min()); d_med_min = tabla6["mediana"].idxmin()
med_max = float(tabla6["mediana"].max()); d_med_max = tabla6["mediana"].idxmax()
iqr_max = float(tabla6["iqr"].max())
max_global = float(compo["rendimiento"].max())
fila_max = compo.loc[compo["rendimiento"].idxmax()]
min_global = float(compo["rendimiento"].min())

texto_f5 = (
    f"El cafe se comporta al reves que los cultivos anteriores. En los seis departamentos la media "
    f"queda por debajo de la mediana (la mayor brecha en {d_gap}: mediana {med_gap:.2f} vs media "
    f"{media_gap:.2f} t/ha). Esta diferencia es consistente con una asimetria hacia rendimientos bajos, "
    f"pero se interpreta con cautela porque el rendimiento reportado esta parcialmente redondeado. Las medianas "
    f"van de {med_min:.2f} ({d_med_min}) a {med_max:.2f} ({d_med_max}), y el rango intercuartilico no "
    f"pasa de {iqr_max:.2f} t/ha. {d_med_max} lidera (mediana {med_max:.2f}) y {d_med_min} queda ultimo "
    f"(mediana {med_min:.2f}). El extremo del corte es {max_global:.2f} t/ha en "
    f"{fila_max['departamento']} ({fila_max['municipio']}, {fila_max['periodo']}) y el minimo cae a "
    f"{min_global:.2f}. El Composite lo muestra todo a la vez: la caja ancha concentrada cerca de 1, "
    f"el rombo de la media a la izquierda de la mediana y puntos que permiten inspeccionar la forma. El "
    f"analisis es descriptivo: no se atribuyen causas."
)
INTERPRETACIONES["fig5_composite"] = texto_f5
print(texto_f5)

print("\n--- Validacion del titulo conclusivo (escrito antes de dibujar) ---")
v_media_menor = bool((tabla6["media"] < tabla6["mediana"]).all())
v_medianas = bool(np.isclose(med_min, 0.95) and np.isclose(med_max, 1.21))
print("  'media < mediana en los 6'      :", v_media_menor,
      "-> brechas (mediana-media):", [round(x, 3) for x in tabla6["brecha_mediana_menos_media"]])
print("  'medianas de 0,95 a 1,21 t/ha' :", v_medianas,
      f"-> medianas [{med_min:.2f}, {med_max:.2f}], IQR max {iqr_max:.2f}")
print("  Titulo sostenido por la tabla:", bool(v_media_menor and v_medianas))

(DIR_HTML.parent / "interpretaciones.json").write_text(
    json.dumps(INTERPRETACIONES, ensure_ascii=False, indent=2), encoding="utf-8")
print("Interpretaciones guardadas en interpretaciones.json")

## Resumen

**Lo que se llevó de este bloque, en una frase:** el boxplot clásico resume una distribución en cinco
números y deja fuera todo lo demás; las cinco variaciones modernas rellenan justo ese hueco, y cada una
gana donde la caja falla.

Las cinco figuras dejaron cinco afirmaciones, y **ninguna se podía leer en el boxplot clásico del mismo
corte**:

- **Half-Eye (papa, 5 departamentos):** la densidad muestra dónde se apila cada departamento; las
  medianas van de 14 a 20 t/ha y la amplitud central del 95 % de 22 a 37 t/ha.
- **Ridge (plátano, 2019–2025):** la forma del rendimiento es estable año tras año, con mediana cerca
  de 7 t/ha; la caja no deja ver si la forma cambia de un año a otro.
- **Sinaplot (cacao, 6 departamentos):** cerca del 44 % de los registros cae en 0,50/0,60 t/ha, una pila
  que la caja esconde tras una mediana aparentemente tranquila.
- **Boxenplot (cinco cultivos con más registros, rendimiento relativo):** normalizada la mediana nacional a 1, la diferencia está en
  las colas: tomate y maíz las tienen pesadas (p99 ≈ 4,4–4,8) y la yuca liviana (p99 ≈ 2,5); la caja,
  y sobre todo su regla de atípicos, no lo muestra.
- **Composite (café, 6 departamentos):** la media es levemente menor que la mediana en los 6; el
  boxplot no lo hace evidente porque nunca dibuja la media
  junto a la mediana.

| Lo que hizo | Con qué |
|-------------|---------|
| Densidad por grupo (Half-Eye) | `gaussian_kde(v)(grid)` recortada al rango observado y dibujada con `go.Scatter(fill="toself")` |
| Bandas de intervalo 66 % y 95 % | `np.quantile(v, [0.17, 0.83])` y `np.quantile(v, [0.025, 0.975])` |
| Ridge por año | una KDE por año sobre un eje x común, desplazadas con `y = i + dens * H` |
| Sinaplot | jitter con ancho proporcional a la densidad: `dens = gaussian_kde(v)(v)` |
| Boxenplot (letter-value) | cajas anidadas `np.quantile(v, [qlo, qhi])` de `(0.25,0.75)` a `(0.0156,0.9844)` |
| Composite Stat-Boxplot | letter-value + `np.median` + `np.mean` (rombo) + puntos, en un solo panel |
| Contraste con la caja | `go.Box(x=v, orientation="h", boxpoints="outliers")` |
| n visible por grupo | texto en la etiqueta del eje: `f"{c}  (n={int(...)})"` |
| Escala declarada | `title_text="... escala lineal"` / `"escala logaritmica"` |
| Orden por mediana | `sort_values(["mediana", "p99"])` (v4: desempate por cola) |
| Exportar | `fig.write_html(..., include_plotlyjs=True)` y `fig.write_image(...)` |

**Las reglas que no se negocian:**

1. El gráfico se deriva de la pregunta, no del gusto ni de lo que trae la librería por defecto.
2. La escala se declara (lineal o logarítmica) y no se cambia entre grupos que se comparan.
3. El n de cada grupo se ve, no se adivina.
4. Las categorías se ordenan por mediana, salvo orden natural (tiempo u ordinal), que se rotula.
5. Todo gris y el color se gana: un elemento resaltado, uno.
6. El color nunca es el único canal: posición, orden o etiqueta lo respaldan.
7. El título dice la conclusión, y la conclusión es verdadera contra la tabla.

**Autoevaluación honesta.** Si puede responder que sí a estas cinco, el bloque quedó cerrado:

- [ ] Puedo decir, para cada una de las cinco variaciones, qué limitación del boxplot corrige.
- [ ] Sé leer de una figura el n, la mediana y un extremo, y encontrarlos en la tabla.
- [ ] Puedo escribir un título que dice una conclusión y verificarlo contra los números.
- [ ] Sé cuándo una escala logarítmica es obligatoria (cocientes) y cuándo es un adorno.
- [ ] Puedo explicar por qué la regla de atípicos de la caja puede engañar en colas pesadas.

**Siguiente:** la presentación, que reúne estas cinco figuras y las cuenta en el orden contexto →
hallazgo → implicación.